<a href="https://colab.research.google.com/github/RnBranco78/iscf-lab1/blob/main/TechnicalReport_SPBD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Technical Report– Taxi Ride Data Analysis (NYC 2013)**
# ***Detecting Anomalous Driver Behavior in NYC Taxi Data***

In [ ]:
#@title Java Setup (needed for pyspark)
!apt-get install -y openjdk-17-jre 2>/dev/null > /dev/null

In [ ]:
#@title Download 1% sample
!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

In [ ]:
#@title Dataset Schema
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]') \
						.appName('taxis').getOrCreate()

try :
    data = spark.read.csv('taxi_rides_1pc.csv.gz', sep =',', header=True, inferSchema=True)

    data.printSchema()

except Exception as err:
    print(err)

root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [ ]:
#Show how many mistakes are( total and by type) and show a row of the ten first mistakes showing.


In [ ]:
# Detecting mistakes in the dataset

from pyspark.sql.functions import col, isnan

# ----- 1. Coordinate mistakes -----
coord_mistakes = data.filter(
    (col("pickup_longitude") == 0) | (col("pickup_latitude") == 0) |
    (col("dropoff_longitude") == 0) | (col("dropoff_latitude") == 0) |
    (col("pickup_longitude") > -50) | (col("pickup_longitude") < -100) |
    (col("dropoff_longitude") > -50) | (col("dropoff_longitude") < -100) |
    (col("pickup_latitude") < 35) | (col("pickup_latitude") > 45) |
    (col("dropoff_latitude") < 35) | (col("dropoff_latitude") > 45)
)

# ----- 2. Monetary mistakes -----
money_cols = ["fare_amount","tip_amount","total_amount","surcharge",
              "mta_tax","tolls_amount"]

money_mistakes = data.filter(
    " OR ".join([f"{c} < 0" for c in money_cols])
)

# ----- 3. Time/Distance mistakes -----
time_dist_mistakes = data.filter(
    (col("trip_time_in_secs") <= 0) | (col("trip_distance") <= 0)
)

# ----- 4. Missing essential fields -----
essential = ["medallion","hack_license","pickup_datetime",
             "dropoff_datetime","trip_time_in_secs","trip_distance"]

missing_mistakes = data.filter(
    " OR ".join([f"{c} IS NULL" for c in essential])
)

# Count mistakes by category
counts = {
    "Invalid coordinates": coord_mistakes.count(),
    "Invalid monetary values": money_mistakes.count(),
    "Invalid time/distance": time_dist_mistakes.count(),
    "Missing essential fields": missing_mistakes.count()
}

# Print counts
print("===== Mistakes by type =====")
for k,v in counts.items():
    print(f"{k}: {v}")

print("\n===== Total mistakes (union of all types) =====")

from functools import reduce
from pyspark.sql import DataFrame

all_mistakes = reduce(DataFrame.unionByName, [
    coord_mistakes,
    money_mistakes,
    time_dist_mistakes,
    missing_mistakes
]).dropDuplicates()

print("Total mistakes:", all_mistakes.count())

# Show the first 10 examples of mistakes
print("\n===== First 10 mistake rows =====")
all_mistakes.show(10, truncate=False)


===== Mistakes by type =====
Invalid coordinates: 35016
Invalid monetary values: 40
Invalid time/distance: 12931
Missing essential fields: 0

===== Total mistakes (union of all types) =====
Total mistakes: 44030

===== First 10 mistake rows =====
+--------------------------------+--------------------------------+-------------------+-------------------+-----------------+-------------+----------------+---------------+-----------------+----------------+------------+-----------+---------+-------+----------+------------+------------+
|medallion                       |hack_license                    |pickup_datetime    |dropoff_datetime   |trip_time_in_secs|trip_distance|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount|
+--------------------------------+--------------------------------+-------------------+-------------------+-----------------+-------------+----------------+---------------+-----

In [ ]:
# Remove invalid rows (master cleaned dataset)

cleaned_data = data.subtract(all_mistakes)

print("Original rows:", data.count())
print("Rows removed:", all_mistakes.count())
print("Rows after cleaning:", cleaned_data.count())

# Quick verification checks
print("Remaining invalid coordinates:", cleaned_data.filter(
    (col("pickup_longitude") == 0) | (col("pickup_latitude") == 0)
).count())

print("Remaining negative monetary values:", cleaned_data.filter(
    " OR ".join([f"{c} < 0" for c in money_cols])
).count())

print("Remaining non-positive time/distance:", cleaned_data.filter(
    (col("trip_time_in_secs") <= 0) | (col("trip_distance") <= 0)
).count())


Original rows: 1735010
Rows removed: 44030
Rows after cleaning: 1690980
Remaining invalid coordinates: 0
Remaining negative monetary values: 0
Remaining non-positive time/distance: 0


In [ ]:
# trip speed calculation
cleaned_data = cleaned_data.withColumn(
    "speed_mph",
    (col("trip_distance") / (col("trip_time_in_secs") / 3600))
)

# tip percentage
cleaned_data = cleaned_data.withColumn(
    "tip_percentage",
    col("tip_amount") / col("fare_amount")
)

# trip hour and day
cleaned_data = cleaned_data.withColumn(
    "hour", hour("pickup_datetime")
).withColumn(
    "weekday", date_format("pickup_datetime", "E")
)


In [ ]:
# Driver-Level Aggregation

from pyspark.sql.window import Window
from pyspark.sql.functions import avg, stddev, count

driver_stats = cleaned_data.groupBy("hack_license").agg(
    count("*").alias("num_trips"),
    avg("trip_distance").alias("avg_distance"),
    avg("trip_time_in_secs").alias("avg_time"),
    avg("speed_mph").alias("avg_speed"),
    stddev("speed_mph").alias("std_speed"),
    avg("tip_amount").alias("avg_tip"),
    avg("tip_percentage").alias("avg_tip_pct")
)

driver_stats_filtered = driver_stats.filter(col("num_trips") >= 30)


In [ ]:
# Baseline Computation

population_baseline = cleaned_data.agg(
    avg("speed_mph").alias("pop_avg_speed"),
    stddev("speed_mph").alias("pop_std_speed"),
    avg("tip_percentage").alias("pop_avg_tip_pct"),
    stddev("tip_percentage").alias("pop_std_tip_pct")
)

population_baseline.show()


#### Convert GPS coordinates to grid cell coordinates

In [ ]:
# Longitude and latitude from the upper left corner of the grid
MIN_LON = -74.916578
MAX_LAT = 41.47718278

# Longitude and latitude that correspond to a shift in 500 meters
LON_DELTA = 0.005986
LAT_DELTA = 0.004491556

def latlon_to_grid(lat, lon):
    return ((int)((MAX_LAT - lat)/LAT_DELTA), (int)((lon - MIN_LON)/LON_DELTA))

#### In Bounds check

You can use cell coordinates to exclude invalid rides

In [ ]:
def inBounds( cell ):
    return cell[0] > 0 and cell[0] < 300 and cell[1] > 0 and cell[1] < 300

still working in this part. Connecting the grid maps with the different drivers




In [ ]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import IntegerType, StructType, StructField

# Grid UDF
@udf("struct<x:int,y:int>")
def latlon_to_grid(lat, lon):
    if lat is None or lon is None:
        return None
    try:
        x = int((MAX_LAT - lat) / LAT_DELTA)
        y = int((lon - MIN_LON) / LON_DELTA)
        return (x, y)
    except:
        return None

data = data.withColumn("pickup_grid", latlon_to_grid(col("pickup_latitude"), col("pickup_longitude")))\
           .withColumn("dropoff_grid", latlon_to_grid(col("dropoff_latitude"), col("dropoff_longitude")))

# Also split into separate ints for easier grouping
data = data.withColumn("pickup_x", col("pickup_grid.x"))\
           .withColumn("pickup_y", col("pickup_grid.y"))\
           .withColumn("dropoff_x", col("dropoff_grid.x"))\
           .withColumn("dropoff_y", col("dropoff_grid.y"))


In [ ]:
driver_grid = data.groupBy("hack_license","pickup_x","pickup_y").count()

driver_grid.show(20, truncate=False)
